# RappiPlus | A/B Testing Analysis

## Project Objective

Evaluate whether the treatment variant produces a statistically significant difference in user conversion compared with the control group.

### Business Question

Does the treatment variant improve user conversion performance?

### Data Source

- `rappiplus_experiment_clean.csv`
- Analysis method: A/B testing
- Primary metric: Conversion rate
- Statistical approach: Two-proportion z-test

### Tools

Python, Pandas, NumPy, Matplotlib, and SciPy.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load cleaned experiment dataset
experiment = pd.read_csv(
    "../data/processed/rappiplus_experiment_clean.csv",
    parse_dates=["timestamp"]
)

print("Dataset shape:", experiment.shape)

display(experiment.head())

Dataset shape: (10000, 7)


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


## 1. Conversion Rate Comparison

### Business Question

Does the treatment variant achieve a higher conversion rate than the control group?

### Methodology

Conversion rates will be calculated for each experimental variant by dividing the number of converted users by the total number of users assigned to that variant.

The analysis will evaluate both the absolute and relative differences in conversion rates before conducting statistical hypothesis testing.

In [2]:
# Compare conversion rates between experimental groups

conversion_summary = (
    experiment.groupby('variante')
    .agg(
        users=('id_usuario', 'nunique'),
        conversions=('convirtio', 'sum'),
        conversion_rate=('convirtio', 'mean')
    )
    .reset_index()
)

# Express conversion rates as percentages
conversion_summary['conversion_rate_pct'] = (
    conversion_summary['conversion_rate'] * 100
)

display(conversion_summary.round(2))

,variante,users,conversions,conversion_rate,conversion_rate_pct
0,control,4965,779,0.16,15.69
1,tratamiento,5035,820,0.16,16.29


In [3]:
# Calculate absolute and relative conversion uplift

control_rate = conversion_summary.loc[
    conversion_summary['variante'] == 'control',
    'conversion_rate'
].iloc[0]

treatment_rate = conversion_summary.loc[
    conversion_summary['variante'] == 'tratamiento',
    'conversion_rate'
].iloc[0]

# Absolute difference in percentage points
absolute_uplift = (treatment_rate - control_rate) * 100

# Relative percentage change
relative_uplift = (
    (treatment_rate - control_rate) / control_rate
) * 100

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Absolute uplift: {absolute_uplift:.2f} percentage points")
print(f"Relative uplift: {relative_uplift:.2f}%")

Control conversion rate: 15.69%
Treatment conversion rate: 16.29%
Absolute uplift: 0.60 percentage points
Relative uplift: 3.80%


## 2. Statistical Hypothesis Testing

### Hypotheses

- **H0 (Null Hypothesis):** The control and treatment variants have equal conversion rates.
- **H1 (Alternative Hypothesis):** The conversion rates differ between the two variants.

### Statistical Method

A two-sided two-proportion z-test will be conducted using a significance level of 5% (α = 0.05).

A 95% confidence interval will also be calculated to estimate the difference between treatment and control conversion rates.

In [4]:
# Two-proportion z-test and 95% confidence interval

from scipy.stats import norm

# Extract group statistics
control = conversion_summary.set_index('variante').loc['control']
treatment = conversion_summary.set_index('variante').loc['tratamiento']

n_control = int(control['users'])
n_treatment = int(treatment['users'])

x_control = int(control['conversions'])
x_treatment = int(treatment['conversions'])

p_control = x_control / n_control
p_treatment = x_treatment / n_treatment

# Observed difference
difference = p_treatment - p_control

# Pooled proportion for hypothesis testing
pooled_p = (
    (x_control + x_treatment)
    / (n_control + n_treatment)
)

# Standard error under the null hypothesis
se_null = np.sqrt(
    pooled_p * (1 - pooled_p)
    * (1 / n_control + 1 / n_treatment)
)

# Z statistic and two-sided p-value
z_stat = difference / se_null
p_value = 2 * norm.sf(abs(z_stat))

# Unpooled standard error for confidence interval
se_diff = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)

# 95% confidence interval (normal approximation)
z_critical = norm.ppf(0.975)

ci_lower = difference - z_critical * se_diff
ci_upper = difference + z_critical * se_diff

alpha = 0.05

print(f"Z statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(
    f"95% CI for difference: "
    f"[{ci_lower * 100:.2f}, {ci_upper * 100:.2f}] percentage points"
)

print(
    "Statistically significant:",
    p_value < alpha
)

Z statistic: 0.8133
P-value: 0.4161
95% CI for difference: [-0.84, 2.03] percentage points
Statistically significant: False


### Key Finding: Statistical Significance

The treatment variant achieved a conversion rate of **16.29%**, compared with **15.69%** for the control group.

This represents an observed absolute increase of **0.60 percentage points** and a relative uplift of **3.80%**.

However, the two-sided two-proportion z-test did not identify a statistically significant difference between the groups:

* **Z-statistic:** 0.8133
* **P-value:** 0.4161
* **Significance level:** 0.05
* **95% confidence interval:** [-0.84, +2.03] percentage points

Since the p-value exceeds 0.05 and the confidence interval includes zero, the null hypothesis cannot be rejected.

### Business Implication

Although the treatment showed a higher observed conversion rate, the available evidence is insufficient to conclude that it improves conversion performance.

### Recommendation

Do not implement the treatment solely based on the observed uplift.

Further experimentation may be warranted, with an appropriate sample-size calculation, a predefined minimum detectable effect, and clearly established success criteria.

The results should not be interpreted as evidence that the two variants are equivalent.


## 3. Sample Ratio Mismatch (SRM) Check

### Objective

Evaluate whether the observed allocation of users between the control and treatment groups is consistent with the expected 50/50 random assignment.

### Methodology

A chi-square goodness-of-fit test will be used to compare the observed group sizes against the expected allocation.

A significance threshold of 0.001 will be used to flag potential sample ratio mismatch.

In [5]:
# Check sample ratio mismatch (SRM)

from scipy.stats import chisquare

# Observed group sizes
observed = conversion_summary['users'].to_numpy()

# Expected 50/50 allocation
expected = np.full(
    len(observed),
    observed.sum() / len(observed)
)

# Chi-square goodness-of-fit test
chi2_stat, srm_p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

# SRM threshold
srm_alpha = 0.001

print(f"Observed group sizes: {observed}")
print(f"Expected group sizes: {expected}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"SRM p-value: {srm_p_value:.4f}")
print(f"Potential SRM detected: {srm_p_value < srm_alpha}")

Observed group sizes: [4965 5035]
Expected group sizes: [5000. 5000.]
Chi-square statistic: 0.4900
SRM p-value: 0.4839
Potential SRM detected: False


## 4. Executive Summary

### Experiment Overview

This analysis evaluated whether the treatment variant improved user conversion compared with the control group using data from 10,000 experimental participants.

### Key Findings

**1. Conversion Performance**

* Control group: 4,965 users and 779 conversions (**15.69%**).
* Treatment group: 5,035 users and 820 conversions (**16.29%**).
* Observed absolute uplift: **+0.60 percentage points**.
* Observed relative uplift: **+3.80%**.

**2. Statistical Significance**

A two-sided two-proportion z-test found no statistically significant difference between the variants.

* Z-statistic: 0.8133
* P-value: **0.4161**
* 95% confidence interval for the treatment-control difference: **[-0.84, +2.03] percentage points**

The confidence interval includes zero, indicating that the observed data are compatible with both a negative and a positive treatment effect.

**3. Sample Ratio Mismatch**

A chi-square goodness-of-fit test found no evidence of sample ratio mismatch under an expected 50/50 allocation.

* SRM p-value: **0.4839**
* SRM threshold: 0.001

### Business Recommendation

**Do not roll out the treatment solely based on the observed conversion uplift.**

The experiment does not provide sufficient statistical evidence that the treatment improves conversion performance.

Before conducting another experiment, define the minimum detectable effect, calculate the required sample size, and establish clear success criteria.

### Analytical Limitations

* The statistical analysis assumes independent observations and appropriate experimental assignment.
* The SRM test evaluates group-size balance but does not establish that randomization was valid in every respect.
* The experiment dataset is analyzed independently of the commercial orders dataset.
* A nonsignificant result does not demonstrate that the variants are equivalent.

### Conclusion

Although the treatment variant achieved a slightly higher observed conversion rate, the difference was not statistically significant.

The appropriate business decision is to avoid claiming an improvement based on the current evidence and evaluate whether additional experimentation is justified.
